# Duration Prediction

In [ ]:
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import root_mean_squared_error

In [ ]:
import mlflow

mlflow.set_experiment("nyc-taxi")

In [ ]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds()/60)
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [ ]:
df = read_dataframe('./data/green_tripdata_2021-01.parquet')

In [ ]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

In [ ]:
dv = DictVectorizer()

train_dicts = df[categorical + numerical].to_dict(orient='records')
x_train = dv.fit_transform(train_dicts)

target = 'duration'
y_train = df[target].values

lr = LinearRegression()
lr.fit(x_train, y_train)

y_pred = lr.predict(x_train)

root_mean_squared_error(y_train, y_pred)

In [ ]:
sns.distplot(y_pred, label='prediction')
sns.distplot(y_train, label='actual')

plt.legend()

In [ ]:
df_train_path = './data/green_tripdata_2021-01.parquet'
df_val_path = './data/green_tripdata_2021-02.parquet'
df_train = read_dataframe(df_train_path)
df_val = read_dataframe(df_val_path)

In [ ]:
len(df_train), len(df_val)

In [ ]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [ ]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
x_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
x_val = dv.transform(val_dicts)

In [ ]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

Linear Regression:
- Result = 7.7587

In [ ]:
with mlflow.start_run():
    mlflow.set_tag("developer", "irb")
    mlflow.set_tag("model", "linear")
    
    mlflow.log_param("train-data-path", df_train_path)
    mlflow.log_param("valid-data-path", df_val_path)

    lr = LinearRegression()
    lr.fit(x_train, y_train)
    
    y_pred = lr.predict(x_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

Lasso:
- 0.001 = 9.2334

In [ ]:
with mlflow.start_run():
    mlflow.set_tag("developer", "irb")
    mlflow.set_tag("model", "lasso")
    
    mlflow.log_param("train-data-path", df_train_path)
    mlflow.log_param("valid-data-path", df_val_path)
    
    alpha = 0.001
    mlflow.log_param("model-alpha", alpha)
    ls = Lasso(alpha)
    ls.fit(x_train, y_train)
    
    y_pred = ls.predict(x_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

Ridge:
- Result = 7.7037

In [ ]:
with mlflow.start_run():
    mlflow.set_tag("developer", "irb")
    mlflow.set_tag("model", "ridge")
    
    mlflow.log_param("train-data-path", df_train_path)
    mlflow.log_param("valid-data-path", df_val_path)
    
    rd = Ridge()
    rd.fit(x_train, y_train)
    
    y_pred = rd.predict(x_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

In [ ]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv,lr), f_out)
with open('models/lasso.bin', 'wb') as f_out:
    pickle.dump((dv,ls), f_out)
with open('models/ridge.bin', 'wb') as f_out:
    pickle.dump((dv,rd), f_out)

# New Experiments

## Running XGBoost trials via Optuna

In [ ]:
import xgboost as xgb
import optuna

In [ ]:
train = xgb.DMatrix(x_train, label=y_train)
valid = xgb.DMatrix(x_val, label=y_val)

In [ ]:
import optuna

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 4, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 0.1, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 0.1, log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1, 20.0, log=True),
        'objective': 'reg:squarederror',
        'seed': 42
    }

    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return rmse

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1) # n_trials is usually set to 50

## Selected Best Parameters from MLflow

In [ ]:
params = {
    'max_depth': 25,
    'learning_rate': 0.09765661001636276,
    'reg_alpha': 0.014519692370291238,
    'reg_lambda': 0.0973254851880949,
    'min_child_weight': 4.9989846475122865,
    'objective': 'reg:squarederror',
    'seed': 42
}

mlflow.xgboost.autolog()
booster = xgb.train(
    params=params,
    dtrain=train,
    num_boost_round=1000,
    evals=[(valid, "validation")],
    early_stopping_rounds=50
)